# Init

In [5]:
# Init

%%time

!pip install sqlalchemy tqdm catboost openai -U "sentence-transformers[train]"

# Clear output

from IPython.display import clear_output

clear_output()

print('Packages installed.')

from google.colab import drive

import os

# if not mounted
if not os.path.exists('/content/drive'):
  drive.mount('/content/drive')

if not os.path.exists('/content/nfs_m.db'):
  !cp -r /content/drive/MyDrive/colab/banshee/data/nfs_m.db /content/
else:
  !echo 'Database already mounted.'

drive.flush_and_unmount()

from sqlalchemy import create_engine

# Create a sqlite engine instance
engine = create_engine(f"sqlite:///nfs_m.db")

import pandas as pd

# Read the database as a dataframe
raw_df = pd.read_sql_table('video_statistics', engine)

raw_df.shape

Packages installed.
Database already mounted.
CPU times: user 10.8 s, sys: 2.61 s, total: 13.4 s
Wall time: 20.1 s


(1150532, 14)

# Preprocessing

In [6]:
# Prep

%%time

from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Input variables

scaler = MinMaxScaler()
captioned_only = False
sample_frac = .1
my_channel_title = 'Mitchell Diedrich'

# Preprocess data

from tqdm.notebook import tqdm

tqdm.pandas

group_data = []
for name, group in tqdm(raw_df.groupby('channel_id')):
  if len(group) > 1:
    row_data = group['video_view_count'].values.reshape(-1, 1)
    # apply target transformations
    group['target'] = scaler.fit_transform(row_data)
    # append to group_data
    group_data.append(group)

# Split data

df = pd.concat(group_data).dropna(subset=['target'])

my_df = df[df['channel_title'] == my_channel_title]
df = df[df['channel_title'] != my_channel_title]

df = df[df['video_caption']==True] if captioned_only else df
df = df.sample(frac=sample_frac, random_state=42)

from sklearn.model_selection import train_test_split

train_df, val_test_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(val_test_df, test_size=0.5, random_state=42)

y_train = train_df['target'].to_numpy()
y_val = val_df['target'].to_numpy()
y_test = test_df['target'].to_numpy()
y_my = my_df['target'].to_numpy()

train_df.shape, val_df.shape, test_df.shape, my_df.shape

  0%|          | 0/1652 [00:00<?, ?it/s]

CPU times: user 7.55 s, sys: 244 ms, total: 7.79 s
Wall time: 6.37 s


((92040, 15), (11505, 15), (11506, 15), (15, 15))

# Generate Embeddings

In [7]:
%%time

# Generate emebeddings

def get_division_count(df, max_len=2048):
  division_count = len(df) / max_len
  return 2 ** (int(division_count).bit_length())

def get_divisions(df):
  division_count = get_division_count(df)
  division_len = len(df) // division_count
  divisions = [df.iloc[i:i+division_len]
               for i in range(0, len(df), division_len)]
  return divisions

def get_embedding_response_data(client, data):
  response = client.embeddings.create(
      input=data,
      model=embedding_model
      )
  return response.data

train_divisions = get_divisions(train_df)
val_divisions = get_divisions(val_df)
test_divisions = get_divisions(test_df)

from google.colab import userdata

openai_key = userdata.get('openai_key')

from openai import OpenAI

embedding_model = "text-embedding-3-large"

client = OpenAI(api_key=openai_key)

import numpy as np

sample_str = str(sample_frac).replace('.', '')
caption_str = 'captionsonly' if captioned_only else 'captionsany'
save_meta_string = f'{sample_str}_{caption_str}_{embedding_model}'

import os

from google.colab import drive

save_dir = '/content/drive/MyDrive/colab/banshee/data/embeddings'

# if not mounted
if not os.path.exists('/content/drive'):
  drive.mount('/content/drive')

train_save_string = f'{save_dir}/{save_meta_string}_train.npy'
if not os.path.exists(train_save_string):
  train_responses = []
  for df in tqdm(train_divisions, desc='Training'):
    division_data = df['video_title'].to_list()
    division_response = get_embedding_response_data(client, division_data)
    division_embeddings = [data.embedding for data in division_response]
    train_responses.extend(division_embeddings)
  train_embeddings = np.array(train_responses)
  np.save(train_save_string, train_embeddings, allow_pickle=True)
train_mb = os.path.getsize(train_save_string) / 1024 / 1024

val_save_string = f'{save_dir}/{save_meta_string}_val.npy'
if not os.path.exists(val_save_string):
  val_responses = []
  for df in tqdm(val_divisions, desc='Validation'):
    division_data = df['video_title'].to_list()
    division_response = get_embedding_response_data(client, division_data)
    division_embeddings = [data.embedding for data in division_response]
    val_responses.extend(division_embeddings)
  val_embeddings = np.array(val_responses)
  np.save(val_save_string, val_embeddings, allow_pickle=True)
val_mb = os.path.getsize(val_save_string) / 1024 / 1024

test_save_string = f'{save_dir}/{save_meta_string}_test.npy'
if not os.path.exists(test_save_string):
  test_responses = []
  for df in tqdm(test_divisions, desc='Test'):
    division_data = df['video_title'].to_list()
    division_response = get_embedding_response_data(client, division_data)
    division_embeddings = [data.embedding for data in division_response]
    test_responses.extend(division_embeddings)
  test_embeddings = np.array(test_responses)
  np.save(test_save_string, test_embeddings, allow_pickle=True)
test_mb = os.path.getsize(test_save_string) / 1024 / 1024

my_save_string = f'{save_dir}/{save_meta_string}_my.npy'
if not os.path.exists(my_save_string):
  my_data = my_df['video_title'].to_list()
  my_response = get_embedding_response_data(client, my_data)
  my_embeddings = np.array([data.embedding for data in my_response])
  np.save(my_save_string, my_embeddings, allow_pickle=True)
my_mb = os.path.getsize(my_save_string) / 1024 / 1024

drive.flush_and_unmount()

train_mb, val_mb, test_mb, my_mb

Mounted at /content/drive
CPU times: user 762 ms, sys: 94.4 ms, total: 856 ms
Wall time: 7.68 s


(2157.1876220703125, 269.6485595703125, 269.6719970703125, 0.3516845703125)

# Copy and Load Embeddings


In [8]:
%%time

import os

from google.colab import drive

if not os.path.exists('/content/drive'):
  drive.mount('/content/drive')

embeddings_path = '/content/drive/MyDrive/colab/banshee/data/embeddings'
files = os.listdir(embeddings_path)

pbar_len = len(train_df) + len(val_df) + len(test_df) + len(my_df)
embed_load_pbar = tqdm(total=pbar_len, desc='Loading embeddings...')

try:
  my_embeddings
except NameError:
  print('my_embeddings not defined, loading...')
  my_embed_file_name =  [f for f in files if 'my' in f][0]
  my_embed_file = os.path.join(embeddings_path, my_embed_file_name)
  my_embeddings = np.load(my_embed_file, allow_pickle=True)
embed_load_pbar.update(len(my_df))

try:
  test_embeddings
except NameError:
  print('test_embeddings not defined, loading...')
  test_embed_file_name =  [f for f in files if 'test' in f][0]
  test_embed_file = os.path.join(embeddings_path, test_embed_file_name)
  test_embeddings = np.load(test_embed_file, allow_pickle=True)
embed_load_pbar.update(len(test_df))

try:
  val_embeddings
except NameError:
  print('val_embeddings not defined, loading...')
  val_embed_file_name =  [f for f in files if 'val' in f][0]
  val_embed_file = os.path.join(embeddings_path, val_embed_file_name)
  val_embeddings = np.load(val_embed_file, allow_pickle=True)
embed_load_pbar.update(len(val_df))

try:
  train_embeddings
except NameError:
  print('train_embeddings not defined, loading...')
  train_embed_file_name =  [f for f in files if 'train' in f][0]
  train_embed_file = os.path.join(embeddings_path, train_embed_file_name)
  train_embeddings = np.load(train_embed_file, allow_pickle=True)
embed_load_pbar.update(len(train_df))

embed_load_pbar.close()

drive.flush_and_unmount()

(train_embeddings.shape, val_embeddings.shape, test_embeddings.shape,
 my_embeddings.shape)

Mounted at /content/drive


Loading embeddings...:   0%|          | 0/115066 [00:00<?, ?it/s]

my_embeddings not defined, loading...
test_embeddings not defined, loading...
val_embeddings not defined, loading...
train_embeddings not defined, loading...
CPU times: user 287 ms, sys: 1.36 s, total: 1.64 s
Wall time: 8.93 s


((92040, 3072), (11505, 3072), (11506, 3072), (15, 3072))

# Model Pool Test

In [9]:
%%time

train_arr = train_df.to_numpy()
train_data = np.concatenate([train_arr, train_embeddings], axis=1)
train_data = pd.DataFrame(train_data, columns=train_df.columns.to_list() +
                          [f'embedding_{i}' for i in range(train_embeddings.shape[1])])

val_arr = val_df.to_numpy()
val_data = np.concatenate([val_arr, val_embeddings], axis=1)
val_data = pd.DataFrame(val_data, columns=val_df.columns.to_list() +
                        [f'embedding_{i}' for i in range(val_embeddings.shape[1])])

test_arr = test_df.to_numpy()
test_data = np.concatenate([test_arr, test_embeddings], axis=1)
test_data = pd.DataFrame(test_data, columns=test_df.columns.to_list() +
                        [f'embedding_{i}' for i in range(test_embeddings.shape[1])])

my_arr = my_df.to_numpy()
my_data = np.concatenate([my_arr, my_embeddings], axis=1)
my_data = pd.DataFrame(my_data, columns=my_df.columns.to_list() +
                        [f'embedding_{i}' for i in range(my_embeddings.shape[1])])

CPU times: user 5.16 s, sys: 7.37 s, total: 12.5 s
Wall time: 12.3 s


In [35]:
train_data = train_df.copy()
train_data['embeddings'] = train_embeddings.tolist()

val_data = val_df.copy()
val_data['embeddings'] = val_embeddings.tolist()

test_data = test_df.copy()
test_data['embeddings'] = test_embeddings.tolist()

my_data = my_df.copy()
my_data['embeddings'] = my_embeddings.tolist()

In [37]:
%%time

from datetime import datetime as dt, timezone as tz

from catboost import CatBoostRegressor, Pool
from tqdm.notebook import tqdm

tqdm.pandas()

def foo(df):

  text_cols = [
    'video_title', 'video_description', 'video_tags', 'video_topic_categories'
  ]
  cat_cols = ['video_caption', 'video_licensed_content']
  num_cols = ['video_duration']
  # Ensure 'embedding' columns contain lists or arrays
  emb_cols = ['embeddings']
  # keep_cols = text_cols + cat_cols + num_cols + emb_cols
  keep_cols = emb_cols
  feature_df = df[keep_cols].copy()

  # for col in cat_cols:
  #   feature_df[col] = df[col].astype(str)

  # str_lambda = lambda x: '' if x is None else x
  # for col in text_cols:
  #   feature_df[col] = feature_df[col].progress_apply(str_lambda)

  target = df['target'].copy()
  # convert times to seconds
  timestamps = pd.to_datetime(df['video_published_at'])
  timestamps = (timestamps - dt(1970, 1, 1, tzinfo=tz.utc)).dt.total_seconds()
  timestamps = (timestamps - timestamps.min()) / (timestamps.max() - timestamps.min())

  pool = Pool(
      data=feature_df,
      label=target.to_numpy(),
      # text_features=text_cols,
      # cat_features=cat_cols,
      embedding_features=emb_cols,
      feature_names=['embeddings']
      # feature_names=feature_df.columns.to_list(),
      # timestamp=timestamps,
  )

  return pool

train_pool = foo(train_data)
val_pool = foo(val_data)
test_pool = foo(test_data)
my_pool = foo(my_data)

CPU times: user 11.5 s, sys: 50.4 ms, total: 11.6 s
Wall time: 11.4 s


In [ ]:
print('start')

model = CatBoostRegressor(
    iterations=128,
    depth=2,
    learning_rate=.1,
    # l2_leaf_reg=10000,
    # random_strength=10000,
    loss_function='RMSE',
    random_seed=42,
    verbose=True,
    task_type='GPU',
    early_stopping_rounds=2048,
    eval_metric='R2',
    metric_period=16,
    )

model.fit(
    train_pool,
    eval_set=val_pool,
    verbose=True
    )

print()
print('Test', model.score(test_pool))
print('My', model.score(my_pool))
print()
print(model.get_feature_importance(prettified=True))

my_pred_df = my_df.copy()
my_pred_df['predicted_views'] = model.predict(my_pool)
my_pred_df = my_pred_df[['video_title', 'video_view_count', 'predicted_views']]
my_pred_df.sort_values(by='predicted_views', ascending=False)

In [11]:
def get_model_info(model, train_pool, val_pool):
  model_info = {
      'timestamp': model.timestamp,
      'iterations': model.get_param('iterations'),
      'depth': model.get_param('depth'),
      'learning_rate': model.get_param('learning_rate'),
      'l2_leaf_reg': model.get_param('l2_leaf_reg'),
      'random_strength': model.get_param('random_strength'),
      'best_iter': model.get_best_iteration(),
      'train_loss': model.get_best_score()['learn']['RMSE'],
      'val_loss': model.get_best_score()['validation']['RMSE'],
      'train_r2': model.perf['train'],
      'val_r2': model.perf['val'],
      'test_r2': model.perf['test'],
      'my_r2': model.perf['my'],
      'runtime': model.perf['runtime']
  }
  return pd.Series(model_info)

grid_info = [get_model_info(model, train_pool, val_pool) for model in models]
grid_df = pd.DataFrame(grid_info)

# convert timestamp to datetime

int_cols = ['iterations', 'depth', 'best_iter']
grid_df[int_cols] = grid_df[int_cols].astype(int)

grid_df['loss_k'] = grid_df['train_loss'] / grid_df['val_loss']
grid_df['r2_k'] = grid_df['val_r2'] / grid_df['train_r2']
grid_df['iter_k'] = (1 + grid_df['best_iter']) / grid_df['iterations']

grid_data = pd.read_csv('grid_data.csv') if os.path.exists('grid_data.csv') else pd.DataFrame()
grid_data = pd.concat([grid_data, grid_df])
grid_data.drop_duplicates(subset=['timestamp'], inplace=True)
grid_data.to_csv('grid_data.csv', index=False)

pd.set_option('display.width', 1000)
print(grid_data)

grid_df.set_index('timestamp', inplace=True)
grid_df.index = pd.to_datetime(grid_df.index, unit='s')
grid_df.sort_values(by='val_r2', ascending=False)

NameError: name 'models' is not defined

In [ ]:
# dont use scientific notation
pd.options.display.float_format = '{:.3f}'.format
temp_my_df = my_df.copy()
temp_my_df['predicted_views'] = model.predict(my_pool)
print(model.score(my_pool))
temp_my_df = temp_my_df[['video_title', 'video_view_count', 'predicted_views']]
temp_my_df.sort_values(by='predicted_views', ascending=False)

\Download thumbs then ask GPT to describe, then embed descriptions.

Also, try classifier (abve median or geomean or *mean*) instead of regressor.

Try unadj view count? Maybe bad idea. Try this as sqrt or log.

Try sqrt or log before adjustment.


Try other scalers (or /avg) as adjusment.

Look at other CatBoost args (category types?, embedding?)

XGBoost? Keras?

incorporate other features (duration, channel data if unadj)

Stop sampling params from grid, try all
